# ProxyML
In this demo, we'll look at how you can use [ProxyML](https://proxyml.ai/) to better understand your machine learning models.  We'll be training a local model to predict whether a potential client is a good or a bad credit risk, and using ProxyML's Python SDK to learn more about our model and our data.

In [1]:
import os
import proxyml
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

In [33]:
# Add your ProxyML API key to the operating environment.  Don't have a key? Grab one for free at https://proxyml.ai/ !
os.environ['PROXYML_API_KEY'] = '<INSERT YOUR KEY HERE>'

## About The Data
We'll be using OpenML's [credit-g](https://www.openml.org/search?type=data&sort=runs&id=31&status=active) (German Credit) dataset for our demo.

In [3]:
credit_data = fetch_openml(name='credit-g', version=1, as_frame=True)
data, target = credit_data.data, credit_data.target
train_data, test_data, train_target, test_target = train_test_split(data, target, test_size=0.1)
print(f"{len(train_data)} sample(s) train, {len(test_data)} sample(s) test")

900 sample(s) train, 100 sample(s) test


In [4]:
display(train_data)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
240,<0,24,existing paid,new car,915,no known savings,>=7,4,female div/dep/mar,none,2,car,29,bank,own,1,skilled,1,none,yes
656,0<=X<200,12,existing paid,new car,888,<100,>=7,4,male single,none,4,car,41,bank,own,1,unskilled resident,2,none,yes
826,<0,18,critical/other existing credit,new car,3966,<100,>=7,1,female div/dep/mar,none,4,real estate,33,bank,rent,3,skilled,1,yes,yes
975,>=200,24,existing paid,radio/tv,1258,500<=X<1000,1<=X<4,3,female div/dep/mar,none,3,car,57,none,own,1,unskilled resident,1,none,yes
561,<0,24,all paid,radio/tv,1546,<100,4<=X<7,4,male single,guarantor,4,car,24,bank,rent,1,unskilled resident,1,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448,>=200,12,existing paid,furniture/equipment,1424,no known savings,>=7,3,female div/dep/mar,none,4,real estate,55,none,own,1,high qualif/self emp/mgmt,1,yes,yes
407,<0,15,existing paid,radio/tv,1053,<100,<1,4,male mar/wid,none,2,real estate,27,none,own,1,skilled,1,none,no
766,<0,30,existing paid,furniture/equipment,3108,<100,<1,2,male div/sep,none,4,life insurance,31,none,own,1,unskilled resident,1,none,yes
633,no checking,9,existing paid,furniture/equipment,1980,<100,<1,2,female div/dep/mar,co applicant,2,car,19,none,rent,2,skilled,1,none,yes


In [5]:
display(train_target)

240     bad
656     bad
826     bad
975    good
561     bad
       ... 
448    good
407    good
766     bad
633     bad
484    good
Name: class, Length: 900, dtype: category
Categories (2, str): ['bad', 'good']

### Feature Selection
We'll use a simple set of features based on the columns in the dataset: we'll one-hot encode the categorical columns, and use the numeric columns as-is.

#### Immutable Columns
Depending on the scenario, there may be some attributes of the data that shouldn't be used for prediction.  In many cases this will also mean that these same attributes shouldn't be used for explaining predictions.  ProxyML supports the exclusion of features by considering them as _immutable_: surrogate models will not use the columns marked as immutable.

For our credit data demo, it makes sense to not consider age or gender in risk assessment.  We'll set the corresponding columns as immutable.

In [6]:
# List of columns we should NOT use for predicting credit risk
immutable_cols = ['age', 'personal_status']  # personal_status encodes gender

# List of numeric cols
numeric_cols = ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
numeric_cols = list(set(numeric_cols) - set(immutable_cols))
display(numeric_cols)

['credit_amount',
 'num_dependents',
 'existing_credits',
 'duration',
 'installment_commitment',
 'residence_since']

In [7]:
# List of columns we'll one-hot encode
categorical_cols = [
    'checking_status', 
    'credit_history', 
    'purpose', 
    'savings_status', 
    'employment', 
    'personal_status', 
    'other_parties', 
    'property_magnitude', 
    'other_payment_plans', 
    'housing', 
    'job', 
    'own_telephone', 
    'foreign_worker'
]

categorical_cols = list(set(categorical_cols) - set(immutable_cols))
display(categorical_cols)

['foreign_worker',
 'savings_status',
 'checking_status',
 'housing',
 'job',
 'purpose',
 'credit_history',
 'employment',
 'other_parties',
 'property_magnitude',
 'own_telephone',
 'other_payment_plans']

In [8]:
# Construct our final "black box" feature set - these are the features that will be made available to our credit risk prediction model.

black_box_feature_cols = list()
black_box_feature_cols.extend(numeric_cols)
black_box_feature_cols.extend(categorical_cols)
display(black_box_feature_cols)

['credit_amount',
 'num_dependents',
 'existing_credits',
 'duration',
 'installment_commitment',
 'residence_since',
 'foreign_worker',
 'savings_status',
 'checking_status',
 'housing',
 'job',
 'purpose',
 'credit_history',
 'employment',
 'other_parties',
 'property_magnitude',
 'own_telephone',
 'other_payment_plans']

Now we'll construct a scikit-learn [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) pipeline that'll one-hot encode the `categorical_cols` and pass through the `numeric_cols`.

In [9]:
# Set up the ColumnTransformer
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), categorical_cols)
    ],
    remainder='passthrough'  # Keeps the numeric columns unchanged
)

# Transform the data
X_train = ct.fit_transform(train_data[black_box_feature_cols])
X_test = ct.transform(test_data[black_box_feature_cols])

y_train = train_target
y_test = test_target

### Training The Local Model
We're ready to train our credit risk predictor.  We'll run a simple cross-validator over a small parameter space for [gradient boosting](https://xgboost.readthedocs.io/en/latest/python/python_api.html#xgboost.XGBClassifier).

In [10]:
scale_pos_weight = (y_train == 'good').sum() / (y_train == 'bad').sum()  # Correct for good/bad imbalance

black_box = RandomizedSearchCV(
    estimator=XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    ),
    param_distributions={
        'n_estimators':     [100, 256, 512],
        'learning_rate':    [0.01, 0.05, 0.1, 0.2],
        'max_depth':        [3, 4, 5, 6],
        'subsample':        [0.6, 0.75, 0.9, 1.0],
        'colsample_bytree': [0.6, 0.75, 0.9, 1.0],
        'min_child_weight': [1, 3, 5],
        'gamma':            [0, 0.1, 0.3, 0.5],
    },
    n_iter=30,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42
)
black_box_le = LabelEncoder()
black_box_ytrain = black_box_le.fit_transform(y_train)
black_box.fit(X_train, black_box_ytrain)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.75, ...], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 4, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1_weighted'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used h

In [11]:
print(f"Best model scored {black_box.best_score_:.2f} with these parameters: {black_box.best_params_}")
black_box_ytest = black_box_le.transform(y_test)
print(f"Score on test data: {black_box.score(X_test, black_box_ytest)}")

Best model scored 0.73 with these parameters: {'subsample': 0.75, 'n_estimators': 100, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.2, 'gamma': 0.1, 'colsample_bytree': 0.75}
Score on test data: 0.7905232974910393


In [12]:
print("Black box classification report:")
print(classification_report(y_test, black_box_le.inverse_transform(black_box.predict(X_test))))

Black box classification report:
              precision    recall  f1-score   support

         bad       0.81      0.45      0.58        29
        good       0.81      0.96      0.88        71

    accuracy                           0.81       100
   macro avg       0.81      0.70      0.73       100
weighted avg       0.81      0.81      0.79       100



## Getting Started With ProxyML
We've got our credit risk predictor, now we'd like to learn more about the predictions it makes.  ProxyML works by training a _surrogate model_ on the basic statistics of your data rather than the data itself, so your data never leaves your server.

### Schema Generation
Your data's statistics are shared with ProxyML by providing a JSON _schema_.  Depending on the size and nature of your data, the schema can be fairly complex but luckily we've got a few options to get you started:

* The ProxyML Python SDK's [`get_schema` function ](https://github.com/proxyml/proxyml-sdk-python/blob/df204c9518ad5135bca1aa9eb1b0dddf95549143/src/proxyml/schema.py#L39) can generate a schema from a pandas DataFrame;
* The [ProxyML Schema Generator page](https://proxyml.github.io/schema-builder/) accepts CSVs and TSVs and generates the schema in your browser; and
* The [ProxyML Schema Generator source code](https://github.com/proxyml/schema-builder) is also available if you'd prefer to run everything yourself.

We'll use the SDK for this demo.

**Note:**  you should consider auto-generated schema as a good starting point, but you should carefully review and adjust the information as needed to best represent your data. It's so important in fact that the SDK includes a note to this effect.  For our demonstration we'll just go with the default schema.

In [13]:
# Creating a ProxyML Schema
surrogate_schema = proxyml.get_schema(train_data, immutable_cols=immutable_cols)
if '_note' in surrogate_schema:
    print(f"Note: {surrogate_schema['_note']}\n")
for feature in surrogate_schema['features']:
    for k, v in feature.items():
        print(f"{k}: {v}")
    print()

Note: Auto-generated schema. Review and adjust types as needed. Integer columns default to count type - consider categorical_ordinal for ordered categories like ratings or class labels.

type: categorical
name: checking_status
valid_categories: {'no checking': 0.3933333333333333, '<0': 0.2777777777777778, '0<=X<200': 0.2688888888888889, '>=200': 0.06}
immutable: False

type: count
name: duration
lambda: 21.04
max: 72
immutable: False

type: categorical
name: credit_history
valid_categories: {'existing paid': 0.5211111111111111, 'critical/other existing credit': 0.29333333333333333, 'delayed previously': 0.09222222222222222, 'all paid': 0.05, 'no credits/all paid': 0.043333333333333335}
immutable: False

type: categorical
name: purpose
valid_categories: {'radio/tv': 0.2822222222222222, 'new car': 0.23333333333333334, 'furniture/equipment': 0.17888888888888888, 'used car': 0.10222222222222223, 'business': 0.09777777777777778, 'education': 0.04888888888888889, 'repairs': 0.021111111111111

With the schema generated, we're ready to share it with ProxyML.  

In [14]:
schema_store_result = proxyml.put_schema(surrogate_schema)
print(schema_store_result)

{'features': [{'name': 'checking_status', 'immutable': False, 'valid_categories': {'no checking': 0.3933333333333333, '<0': 0.2777777777777778, '0<=X<200': 0.2688888888888889, '>=200': 0.06}, 'type': 'categorical'}, {'name': 'duration', 'immutable': False, 'lambda': 21.04, 'max': 72, 'type': 'count'}, {'name': 'credit_history', 'immutable': False, 'valid_categories': {'existing paid': 0.5211111111111111, 'critical/other existing credit': 0.29333333333333333, 'delayed previously': 0.09222222222222222, 'all paid': 0.05, 'no credits/all paid': 0.043333333333333335}, 'type': 'categorical'}, {'name': 'purpose', 'immutable': False, 'valid_categories': {'radio/tv': 0.2822222222222222, 'new car': 0.23333333333333334, 'furniture/equipment': 0.17888888888888888, 'used car': 0.10222222222222223, 'business': 0.09777777777777778, 'education': 0.04888888888888889, 'repairs': 0.021111111111111112, 'domestic appliance': 0.013333333333333334, 'other': 0.012222222222222223, 'retraining': 0.01}, 'type': 

## Data Synthesis
The next step in our process is to generate synthetic data, which are based on the descriptive statistics we just uploaded.  If we provide these synthetic data to our credit risk model, it can make risk predictions...and we can train a surrogate to predict its risk prediction by pairing the synthetic data with the risk predictions.  That's how ProxyML provides explainability without ever actually seeing your real data.

ProxyML's `synthesize_data` has two modes of operation: _neighbors_ and _blended_.  In neighbors (also known as "global") mode, synthesis is strictly based on a random sample of the synthetic data's featurespace.  For example, if your data has a continuous numeric feature `Retail Sq. Ft` with mean $\mu$ and standard deviation $\sigma$, every synthetic sample will have a `Retail Sq. Ft` feature drawn from `np.random.normal(loc=`$\mu$ `, scale=` $\sigma$ `)` .

In blended mode, synthesis is based on a blend of global sampling and local pertubations around a provided sample, provided in your call to `synthesize_data`.  This is particularly useful for "what if?" counterfactual scenarios - in our current credit risk problem for example we might be interested to see what would have to change for a `bad` risk to change to a `good` risk.  This does require sharing a sample with ProxyML, but it doesn't necessarily have to be a real sample - you could for example generate your own synthetic sample that was a good match for the real sample, or simply generate an initial batch of global synthetic samples with ProxyML and then choose the synthetic sample that most closely matches your real sample.

To keep things simple for this demo, we'll stick to global mode.

In [15]:
# Synthesize data
synth_df = proxyml.synthesize_data(num_points=500)
print(f"\nSynthesized {len(synth_df)} rows with columns: {list(synth_df.columns)}")
display(synth_df)


Synthesized 500 rows with columns: ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker']


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,no checking,26,critical/other existing credit,used car,3235,no known savings,unemployed,4,male single,none,3,car,35,none,rent,0,skilled,0,none,yes
1,no checking,27,existing paid,used car,3282,<100,4<=X<7,4,female div/dep/mar,none,4,life insurance,36,none,own,0,skilled,0,yes,yes
2,<0,28,delayed previously,repairs,3347,<100,>=7,1,female div/dep/mar,guarantor,2,life insurance,43,none,rent,0,unskilled resident,2,none,yes
3,no checking,18,existing paid,new car,3300,<100,1<=X<4,1,male single,none,4,life insurance,34,bank,rent,1,skilled,2,yes,yes
4,<0,15,existing paid,furniture/equipment,3316,<100,4<=X<7,4,female div/dep/mar,none,3,car,31,none,own,0,skilled,2,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0<=X<200,16,critical/other existing credit,furniture/equipment,3311,500<=X<1000,<1,3,female div/dep/mar,none,3,real estate,39,none,own,4,unskilled resident,0,none,yes
496,<0,21,existing paid,radio/tv,3307,500<=X<1000,1<=X<4,4,female div/dep/mar,guarantor,1,life insurance,30,none,rent,1,skilled,1,none,no
497,0<=X<200,24,existing paid,radio/tv,3317,<100,1<=X<4,4,male single,none,1,real estate,35,none,own,0,high qualif/self emp/mgmt,1,yes,no
498,no checking,28,existing paid,radio/tv,3359,100<=X<500,>=7,4,female div/dep/mar,none,4,car,37,none,own,3,skilled,2,none,yes


## Surrogate Model
We're ready to train our surrogate model! All we need are the synthetic data we just generated, and our credit risk model's predictions for these synthetic data.

In [16]:
# Train a surrogate model
black_box_predictions = black_box_le.inverse_transform(black_box.predict(ct.transform(synth_df[black_box_feature_cols]))).tolist()
train_result = proxyml.train_surrogate(
    samples=synth_df.values.tolist(),
    predictions=black_box_predictions,
    feature_names=list(synth_df.columns),
    task="classification",  # "classification," "regression," or "auto" in which case ProxyML will try to programatically determine the type of task
    test_size=0.2,
)
print("\nSurrogate training result:")
for k, v in train_result.items():
    print(f"{k}: {v}")


Surrogate training result:
version: 8c59a35a-9a14-4690-af59-368af197da5e
trained_at: 2026-04-23 14:17:17
task: classification
schema_name: default
name: None
comments: None
feature_names: ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'other_parties', 'residence_since', 'property_magnitude', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker']
metrics: {'f1': 0.8769200779727094, 'accuracy': 0.86}


We can retrieve additional details about our surrogate, for example a summary of our new surrogate:

In [17]:
surrogate_summary = proxyml.get_model_summary(version=train_result['version'])
surrogate_summary

{'model_version': '8c59a35a-9a14-4690-af59-368af197da5e',
 'task': 'classification',
 'trained_at': '2026-04-23 14:17:17',
 'name': None,
 'comments': None,
 'feature_names': ['checking_status',
  'duration',
  'credit_history',
  'purpose',
  'credit_amount',
  'savings_status',
  'employment',
  'installment_commitment',
  'other_parties',
  'residence_since',
  'property_magnitude',
  'other_payment_plans',
  'housing',
  'existing_credits',
  'job',
  'num_dependents',
  'own_telephone',
  'foreign_worker'],
 'metrics': {'f1': 0.8769200779727094, 'accuracy': 0.86},
 'feature_importances': [{'feature': 'purpose',
   'coefficient': 6.245230870250408,
   'abs_coefficient': 6.245230870250408},
  {'feature': 'foreign_worker',
   'coefficient': 5.752953030793057,
   'abs_coefficient': 5.752953030793057},
  {'feature': 'savings_status',
   'coefficient': 4.23126628395648,
   'abs_coefficient': 4.23126628395648},
  {'feature': 'other_payment_plans',
   'coefficient': 2.7975009195749223,
  

Or just the feature importances:

In [18]:
surrogate_importances = proxyml.get_feature_importances(version=train_result['version'])
surrogate_importances

{'feature_importances': [{'feature': 'purpose',
   'coefficient': 6.245230870250408,
   'abs_coefficient': 6.245230870250408},
  {'feature': 'foreign_worker',
   'coefficient': 5.752953030793057,
   'abs_coefficient': 5.752953030793057},
  {'feature': 'savings_status',
   'coefficient': 4.23126628395648,
   'abs_coefficient': 4.23126628395648},
  {'feature': 'other_payment_plans',
   'coefficient': 2.7975009195749223,
   'abs_coefficient': 2.7975009195749223},
  {'feature': 'housing',
   'coefficient': 2.768176184590004,
   'abs_coefficient': 2.768176184590004},
  {'feature': 'checking_status',
   'coefficient': 2.3162644059949344,
   'abs_coefficient': 2.3162644059949344},
  {'feature': 'credit_history',
   'coefficient': 2.2519775447147206,
   'abs_coefficient': 2.2519775447147206},
  {'feature': 'duration',
   'coefficient': -1.650951149483543,
   'abs_coefficient': 1.650951149483543},
  {'feature': 'employment',
   'coefficient': 1.5791434434295568,
   'abs_coefficient': 1.57914344

And of course we can make predictions with our surrogate, and compare with our credit risk model.  Note that the surrogate won't always agree with your "real" model right off the bat, you may need to to experiment with schema and synthetic data to get the best result.

In [19]:
sample_point = synth_df.sample()
xformed_sample = ct.transform(sample_point)
display(sample_point)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
450,no checking,19,critical/other existing credit,repairs,3323,no known savings,4<=X<7,1,female div/dep/mar,none,4,life insurance,37,none,rent,1,skilled,0,yes,yes


In [21]:
# Surrogate prediction - batch mode
proxyml.predict_batch(sample_point.values.tolist())

{'predictions': ['good'],
 'model_version': 'surrogate-8c59a35a-9a14-4690-af59-368af197da5e-classification',
 'probabilities': [[5.773159728050814e-15, 0.9999999999999942]]}

In [22]:
# Surrogate prediction - single sample
proxyml.predict(sample=sample_point.values.tolist()[0])

{'prediction': 'good',
 'model_version': 'surrogate-8c59a35a-9a14-4690-af59-368af197da5e-classification',
 'probabilities': [5.773159728050814e-15, 0.9999999999999942]}

In [21]:
# Compare with our credit risk predictor
print(f"Credit risk prediction: '{black_box_le.inverse_transform(black_box.predict(xformed_sample))[0]}', probabilities {black_box.predict_proba(xformed_sample)[0]}")

Credit risk prediction: 'good', probabilities [0.07915324 0.92084676]


#### Local Explanations
We can also retrieve an _explained_ prediction for a sample, which details how the attributes of the sample contributed to our surrogate's prediction.  ProxyML uses linear surrogates so these contributions are exact: the sum of contributions for a given sample plus the intercept equals the raw model output (the predicted value for regression, the decision function for classification).

In [25]:
# Surrogate prediction - single sample with local explanation
explained_prediction_result = proxyml.explain_local(sample_point.values.tolist()[0])
prediction = explained_prediction_result['prediction']
probs = explained_prediction_result['probabilities']

print(f"Prediction: '{prediction}' with probabilities {probs}")
print("\nContribution to prediction by feature:")
for feature_contribution in explained_prediction_result['feature_contributions']:
    feature_name = feature_contribution['feature']
    contribution = feature_contribution['contribution']
    print(f"\t{feature_name}: {contribution:.4f}")

Prediction: 'good' with probabilities [5.773159728050814e-15, 0.9999999999999942]

Contribution to prediction by feature:
	purpose: 13.0521
	foreign_worker: -4.8517
	checking_status: 4.4500
	credit_history: 4.3051
	employment: 3.9952
	housing: -3.2510
	savings_status: 1.9041
	other_payment_plans: -1.3382
	own_telephone: 1.0090
	num_dependents: 0.7327
	job: -0.6984
	other_parties: 0.5871
	duration: 0.5748
	residence_since: 0.4762
	installment_commitment: 0.3941
	property_magnitude: -0.2714
	credit_amount: -0.2129
	existing_credits: -0.0030


While we're at it, we should probably take a quick look at our usage and our available surrogates and schemas.

In [26]:
print(proxyml.get_usage())

{'tier': 'pro', 'period': '2026-04', 'calls_this_period': 76, 'calls_limit': 10000, 'calls_remaining': 9924, 'surrogates_trained': 1, 'surrogate_limit': None}


In [27]:
print("Surrogate Models")
for surrogate in proxyml.list_models():
    print(surrogate)

print()
print("Data Schema")
for schema in proxyml.list_schemas():
    print(schema)

Surrogate Models
{'version': '8c59a35a-9a14-4690-af59-368af197da5e', 'trained_at': '2026-04-23 14:17:17', 'task': 'classification', 'schema_name': 'default', 'name': None, 'comments': None, 'feature_names': ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'other_parties', 'residence_since', 'property_magnitude', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker'], 'metrics': {'f1': 0.8769200779727094, 'accuracy': 0.86}}

Data Schema
{'name': 'default', 'updated_at': '2026-04-23 14:16:57', 'feature_names': ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker'], 'surr

## Counterfactuals
Let's take a look at counterfactuals in more detail.  Suppose we're interested in seeing what it would take to get some of the bad credit risks to become good credit risks.

In [28]:
xformed_synthetic_data = ct.transform(synth_df)
black_box_predictions = black_box_le.inverse_transform(black_box.predict(xformed_synthetic_data))

bad_risk_samples = (
    synth_df[black_box_predictions == 'bad']
    .head(5)
    .apply(lambda x: x.map(lambda v: v.item() if hasattr(v, 'item') else v))
    .values.tolist()
)

Now we can send these predicted _bad_ risks to ProxyML, and have it return counterfactuals. We'll look at each feature of each sample and identify which feature(s) could be changed to potentially change the risk to _good_.

It's worth noting that not every counterfactual will automatically flip the risk prediction. In these cases it's also worth examining the predicted probabilities for both the original sample and the counterfactual — ideally, even if the label itself didn't change, the probabilities will have shifted toward the desired outcome. This would still provide evidence that the suggested changes moved the model closer to the desired prediction, even if the decision boundary wasn't fully crossed.  In these cases, increasing `n_neighbors` or adjusting `perturbation_scale` are worth considering.

In [31]:
cfs = proxyml.find_counterfactuals(
    samples=bad_risk_samples,
    target='good',
    n_neighbors=10000,
    perturbation_scale=0.25
)

for i, cf in enumerate(cfs):
    sample_df = pd.DataFrame([bad_risk_samples[i]], columns=cf.columns)
    diff = sample_df.iloc[0].compare(cf.iloc[0], result_names=('Original', 'Counterfactual'))
    print(f"Sample {i + 1}:")
    display(diff)
    print(f"Original credit risk prediction: '{black_box_le.inverse_transform(black_box.predict(ct.transform(sample_df)))[0]}' ({black_box.predict_proba(ct.transform(sample_df))[0]})")
    print(f"Counterfactual credit risk prediction: '{black_box_le.inverse_transform(black_box.predict(ct.transform(cf)))[0]}' ({black_box.predict_proba(ct.transform(cf))[0]})")
    print()

Sample 1:


,Original,Counterfactual
housing,rent,own


Original credit risk prediction: 'bad' ([0.8485507  0.15144932])
Counterfactual credit risk prediction: 'bad' ([0.54059386 0.45940617])

Sample 2:


,Original,Counterfactual
purpose,new car,radio/tv


Original credit risk prediction: 'bad' ([0.5468435  0.45315647])
Counterfactual credit risk prediction: 'good' ([0.44131988 0.5586801 ])

Sample 3:


...

Original credit risk prediction: 'bad' ([0.7841265  0.21587346])
Counterfactual credit risk prediction: 'good' ([0.05608582 0.9439142 ])

Sample 4:


,Original,Counterfactual
purpose,education,used car


Original credit risk prediction: 'bad' ([0.83066964 0.16933036])
Counterfactual credit risk prediction: 'good' ([0.24541068 0.7545893 ])

Sample 5:


,Original,Counterfactual
credit_history,existing paid,critical/other existing credit


Original credit risk prediction: 'bad' ([0.5524728  0.44752723])
Counterfactual credit risk prediction: 'good' ([0.35859537 0.6414046 ])



The API also has an `interpret_counterfactual()` function that you can use to get a text summarization of what changed between original and counterfactual without making an API call.

In [32]:
for i, cf in enumerate(cfs):
    sample_df = pd.DataFrame([bad_risk_samples[i]], columns=cf.columns)
    cf_interp = proxyml.interpret_counterfactual(
        sample=sample_df.to_dict(orient='records')[0],
        counterfactual=cf.to_dict(orient='records')[0],
        prediction_changed=black_box.predict(ct.transform(sample_df))[0] != black_box.predict(ct.transform(cf))[0],
        exclude_from_diff=immutable_cols
    )
    print(f"Sample {i + 1}:  {cf_interp}\n")

Sample 1:  A counterfactual was found suggesting housing from rent to own, however the original model's prediction did not change. This may indicate the surrogate model does not fully capture the original model's decision boundary in this region.

Sample 2:  Changing purpose from new car to radio/tv may result in a different prediction. Note: this is based on a surrogate model and should be treated as approximate.

Sample 3:  Changing purpose from new car to used car may result in a different prediction. Note: this is based on a surrogate model and should be treated as approximate.

Sample 4:  Changing purpose from education to used car may result in a different prediction. Note: this is based on a surrogate model and should be treated as approximate.

Sample 5:  Changing credit_history from existing paid to critical/other existing credit may result in a different prediction. Note: this is based on a surrogate model and should be treated as approximate.



Your results may vary, but for this particular run we observed that changing one or two features in a _bad_ credit risk was often sufficient to flip them to being a _good_ credit risk.  We saw one sample that wasn't able to move to _good_ risk; however the probabilities did move in the right direction.

Some notes on what we've seen during various runs of this demonstration:

* Having a checking account with a positive balance vs. a negative balance decreased risk;
* Some loan purposes appear riskier than others e.g. business loans appear to be riskier than used car loans and new car loans are riskier than radio/tv loans;
* Savings status showed that a balance of 100DM or less was _more_ risk than `no known savings` however in this dataset `no known savings` consists of both "_no_ savings" and "_unknown_ savings," suggesting that some applicants may in fact have unaccounted assets.
* Counterintuitive results for `credit_history`: in this particular dataset `all paid` includes people with no credit history, suggesting that even a problematic history is better than no history at all.

Remember that none of these counterfactuals required access to the original training data or the internals of the black box model!  We only needed the surrogate model's approximation of the black box's behavior on synthetic data.